In [157]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

In [158]:
def probable_role(df):

    roles = {}
    date_pattern = r"^(\d{4}[-/.]\d{1,2}[-/.]\d{1,2}|\d{1,2}[-/.]\d{1,2}[-/.]\d{4})(\s+\d{1,2}:\d{2}(:\d{2})?)?$"

    for col in df.columns:

        if (df[col] == 0).all() or df[col].isna().all():
            roles[col] = "Unknown or mixed type"
            continue

        if df[col].dropna().dtype == 'bool':
            roles[col] = "Boolean"
            continue

        if len(df[col].dropna().unique()) == 2:
                roles[col] = "Boolean"
                continue

        if pd.api.types.is_numeric_dtype(df[col].dropna()):
            roles[col] = "Numeric"
            continue

        if pd.api.types.is_datetime64_any_dtype(df[col].dropna()):
            roles[col] = "Date-like"
            continue

        if df[col].dropna().astype(str).str.match(date_pattern).mean() >= 0.8:
            roles[col] = "Date-like"
            continue

        if len(df[col].dropna().unique())/len(df[col].dropna()) > 0.90:
            roles[col] = "Identifier-like"
            continue

        if (len(df[col].dropna().unique()) > 2) & (len(df[col].dropna().unique())/len(df[col].dropna()) < 0.05):
            roles[col] = "Categorical"
            continue

        if pd.api.types.is_object_dtype(df[col].dropna()) or pd.api.types.is_string_dtype(df[col].dropna()):
            roles[col] = "Free-text field"
            continue
            
        else:
            roles[col] = "Unknown or mixed type"
   
    return roles


In [159]:
def get_dataset_overview(df, dataset_name, roles):
    overview = {
        'filename': dataset_name,
        'rows': len(df),
        'columns': len(df.columns),
        'column_name': {},
        'technical_type': {},
        'non_missing_count': {},
        'missing_percent': {},
        'unique_values': {},
        'probable_role': {},
    }
    for col in df.columns:
        missing_count = int(df[col].isna().sum())
        overview['column_name'] = col
        overview['technical_type'][col] = str(df[col].dtype)
        overview['non_missing_count'][col] = len(df[col]) - missing_count
        overview['missing_percent'][col] = round(missing_count / len(df[col]), 4)
        overview['unique_values'][col] = int(df[col].dropna().nunique())
        overview['probable_role'][col] = roles[col]
    return overview

In [164]:
def get_data_quality_profile(df, roles):
    
    missing_threshold = 0.30
    sensitive_keywords = ['ssn', 'password', 'credit', 'phone', 'email',
                           'address', 'zip', 'dob', 'birth', 'salary', 'income', 'social_security']

    profile = {
        'Number of duplicate rows in each columns': {},
        'Columns containing only one distinct value': [],
        'Columns with a high percentage of missing values': [],
        'Columns that have mixed or inconsistent data type': [col for col, role in roles.items() if role == "Unknown or mixed type"],
        'Columns that have identifier-like or extremely high-cardinality data type': [col for col, role in roles.items() if role == "Identifier-like"],
        'Columns that might contain sensitive data based on column name': []
    }

    for col in df.columns:
        profile['Number of duplicate rows in each columns'][col] = int(df.dropna().duplicated().sum())
        if df[col].dropna().nunique() == 1:
            profile['Columns containing only one distinct value'].append(col)
        if round(int(df[col].isna().sum()) / len(df[col]), 4) >= missing_threshold:
            profile['Columns with a high percentage of missing values'].append(col)
        if any(keyword in col.lower() for keyword in sensitive_keywords):
            profile['Columns that might contain sensitive data based on column name'].append(col)
            
    return profile

In [180]:
def descriptive_statistics(df, roles):

    Categorical_Columns = [col for col, role in roles.items() if role == "Categorical"]
    Categorical_Columns_Stats = {
        'Number of unique categories in the column': {},
        'Most frequent category or categories': {},
        'Frequency count of top 10 categories': {},
        'Frequency count percentage of top 10 categories': {},
        'Top categories when the column contains many unique values': {}
    }

    Numeric_Columns = [col for col, role in roles.items() if role == "Numeric"]
    Numeric_Columns_Stats = {
        'Valid count of the column': {},
        'Missing count and percentage of the column': {},
        'Minimum value in the column': {},
        'Maximum value in the column': {},
        'Mean of the column': {},
        'Median of the column': {},
        'Mode of the column': {},
        'Standard deviation of the column': {},
        'First quartile of the column': {},
        'Third quartile of the column': {},
        'Interquartile of the column': {},
        'Outlier count and percentage of the column': {}
    }

    Stats = {
        'Categorical Columns descriptive statistics (if blank then no categorical columns found)': Categorical_Columns_Stats, 
        'Numerical Columns descriptive statistics (if blank then no numeric columns found)': Numeric_Columns_Stats
    }

    for col in Categorical_Columns:
        Categorical_Columns_Stats['Number of unique categories in the column'][col] = int(df[col].nunique())
        Categorical_Columns_Stats['Most frequent category or categories'][col] = df[col].value_counts().head(3)

        Categorical_Columns_Stats['Frequency count of top 10 categories'][col] = {}
        Categorical_Columns_Stats['Frequency count percentage of top 10 categories'][col] = {}

        for category, count in df[col].value_counts().head(10).items():
            Categorical_Columns_Stats['Frequency count of top 10 categories'][col][category] = int(count)
            Categorical_Columns_Stats['Frequency count percentage of top 10 categories'][col][category] = round((count / len(df[col]))*100,1)
            
        if df[col].nunique() > 10:
            Categorical_Columns_Stats['Top categories when the column contains many unique values'][col] = df[col].value_counts().head(3)

    for col in Numeric_Columns:
        valid_data = df[col].dropna()
        Numeric_Columns_Stats['Valid count of the column'][col] = len(valid_data)
        Numeric_Columns_Stats['Missing count and percentage of the column'][col] = {len(df[col].isna()), round((len(df[col].isna())/len(df[col]))*100,1)}
        Numeric_Columns_Stats['Minimum value in the column'][col] = float(valid_data.min())
        Numeric_Columns_Stats['Maximum value in the column'][col] = float(valid_data.max())
        Numeric_Columns_Stats['Mean of the column'][col] = float(valid_data.mean())
        Numeric_Columns_Stats['Median of the column'][col] = float(valid_data.median())
        Numeric_Columns_Stats['Mode of the column'][col] = float(valid_data.mode().iloc[0])
        Numeric_Columns_Stats['Standard deviation of the column'][col] = float(valid_data.std())
        q1 = valid_data.quantile(0.25)
        q3 = valid_data.quantile(0.75)
        iqr = q3 - q1
        outliers = valid_data[(valid_data < (q1 - 1.5 * iqr)) | (valid_data > (q3 + 1.5 * iqr))]
        Numeric_Columns_Stats['First quartile of the column'][col] = float(q1)
        Numeric_Columns_Stats['Third quartile of the column'][col] = float(q3)
        Numeric_Columns_Stats['Interquartile of the column'][col] = float(iqr)
        Numeric_Columns_Stats['Outlier count and percentage of the column'][col] = {int(len(outliers)), round((int(len(outliers))/len(df[col]))*100,1)}

    return Stats



In [178]:
csv_path = "../data/Provisional COVID-19 data.csv"
def generate_profile(csv_path):
    dataset_name = os.path.basename(csv_path)
    df2 = pd.read_csv(csv_path)
    probable_role_output = probable_role(df2)
    get_dataset_overview_output = get_dataset_overview(df2, dataset_name, probable_role_output)
    get_data_quality_profile_output = get_data_quality_profile(df2, probable_role_output)
    descriptive_statistics_output = descriptive_statistics(df2, probable_role_output)

    return get_dataset_overview_output, get_data_quality_profile_output, descriptive_statistics_output

generate_profile(csv_path)

({'filename': 'Provisional COVID-19 data.csv',
  'rows': 45565,
  'columns': 14,
  'column_name': 'footnote',
  'technical_type': {'data_as_of': 'str',
   'Jurisdiction_Residence': 'str',
   'Group': 'str',
   'data_period_start': 'str',
   'data_period_end': 'str',
   'COVID_deaths': 'str',
   'COVID_pct_of_total': 'float64',
   'pct_change_wk': 'float64',
   'pct_diff_wk': 'float64',
   'crude_COVID_rate': 'float64',
   'aa_COVID_rate': 'float64',
   'crude_COVID_rate_ann': 'str',
   'aa_COVID_rate_ann': 'str',
   'footnote': 'str'},
  'non_missing_count': {'data_as_of': 45565,
   'Jurisdiction_Residence': 45565,
   'Group': 45565,
   'data_period_start': 45565,
   'data_period_end': 45565,
   'COVID_deaths': 38958,
   'COVID_pct_of_total': 38958,
   'pct_change_wk': 13691,
   'pct_diff_wk': 14908,
   'crude_COVID_rate': 36006,
   'aa_COVID_rate': 36006,
   'crude_COVID_rate_ann': 36006,
   'aa_COVID_rate_ann': 36006,
   'footnote': 9559},
  'missing_percent': {'data_as_of': 0.0,
   

In [ ]:
def get_relationships_between_variables(df, roles):
    """Calculates correlations between appropriate numeric columns."""
    relationships = {}
    numeric_cols = [col for col, info in roles.items() if info['role'] == 'Numeric']
    
    if len(numeric_cols) > 1:
        corr_matrix = df[numeric_cols].corr()
        corr_unstacked = corr_matrix.abs().unstack()
        corr_unstacked = corr_unstacked[corr_unstacked < 1.0].dropna()
        
        if not corr_unstacked.empty:
            strongest = corr_unstacked.idxmax()
            relationships['strongest_pair'] = strongest
            relationships['correlation_value'] = round(corr_matrix.loc[strongest[0], strongest[1]], 4)
            relationships['skipped'] = False
    else:
        relationships['skipped'] = True
        relationships['reason'] = "Not enough numeric columns for correlation analysis."
        
    return relationships

In [ ]:
def generate_adaptive_visualizations(df, roles, plots_dir):
    """Selects and generates plots based on available attribute types."""
    generated_plots = []
    sns.set_theme(style="whitegrid")
    
    # 1. Missing Values Bar Chart
    missing_data = df.isna().sum()
    if missing_data.sum() > 0:
        plt.figure(figsize=(10, 6))
        missing_data[missing_data > 0].plot(kind='bar', color='salmon')
        plt.title('Missing Values per Column')
        plt.ylabel('Count of Missing Values')
        plt.tight_layout()
        plt.savefig(os.path.join(plots_dir, 'plot_01_missing_values.png'))
        plt.close()
        generated_plots.append('plot_01_missing_values.png')

    numeric_cols = [col for col, info in roles.items() if info['role'] == 'Numeric']
    cat_cols = [col for col, info in roles.items() if info['role'] in ['Categorical/Text', 'Boolean']]
    plot_num = len(generated_plots) + 1

    # 2. Numeric Correlation Heatmap
    if len(numeric_cols) >= 2:
        plt.figure(figsize=(10, 8))
        sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
        plt.title('Numeric Correlation Heatmap')
        plt.tight_layout()
        plt.savefig(os.path.join(plots_dir, f'plot_{plot_num:02d}_correlation.png'))
        plt.close()
        generated_plots.append(f'plot_{plot_num:02d}_correlation.png')
        plot_num += 1

    # 3. Numeric Distributions (Histograms)
    for col in numeric_cols[:3]: 
        plt.figure(figsize=(10, 6))
        sns.histplot(df[col].dropna(), kde=True, bins=30)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.tight_layout()
        filename = f'plot_{plot_num:02d}_dist_{col[:10]}.png'
        plt.savefig(os.path.join(plots_dir, filename))
        plt.close()
        generated_plots.append(filename)
        plot_num += 1

    # 4. Categorical Frequencies (Bar Charts)
    for col in cat_cols[:3]:
        plt.figure(figsize=(10, 6))
        df[col].value_counts().head(10).plot(kind='bar', color='skyblue')
        plt.title(f'Top Categories in {col}')
        plt.ylabel('Frequency')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        filename = f'plot_{plot_num:02d}_cat_{col[:10]}.png'
        plt.savefig(os.path.join(plots_dir, filename))
        plt.close()
        generated_plots.append(filename)
        plot_num += 1

    return generated_plots

In [ ]:
def run_llm_insights(summary, output_dir, use_llm):
    """Passes structured summary to local LLM with graceful fallback."""
    prompt = f"""
    You are an expert data analyst. I have profiled a dataset and generated the following verified statistics in JSON format. 
    Using ONLY these facts, write 5 to 8 evidence-based insights. 
    
    You must include:
    - At least one data-quality finding.
    - At least one distribution finding.
    - At least one categorical finding.
    - At least one relationship finding (if applicable).
    - At least one limitation or potential-bias warning.
    - At least one recommended question for further investigation.

    Data Summary:
    {json.dumps(summary, indent=2)}

    Do not calculate new statistics. Do not invent facts. Write clearly and professionally.
    """
    
    with open(os.path.join(output_dir, 'llm_prompt.txt'), 'w', encoding='utf-8') as f: f.write(prompt)

    if not use_llm:
        msg = "LLM insights skipped (use_llm=False)."
        with open(os.path.join(output_dir, 'llm_response.txt'), 'w', encoding='utf-8') as f: f.write(msg)
        return msg

    try:
        response = requests.post('http://localhost:11434/api/generate', json={"model": "llama3", "prompt": prompt, "stream": False}, timeout=45)
        response.raise_for_status()
        insights = response.json().get('response', '')
        with open(os.path.join(output_dir, 'llm_response.txt'), 'w', encoding='utf-8') as f: f.write(insights)
        return insights
    except Exception as e:
        msg = f"Graceful Failure: LLM unavailable.\nError: {e}"
        with open(os.path.join(output_dir, 'llm_response.txt'), 'w', encoding='utf-8') as f: f.write(msg)
        return msg

In [ ]:
def build_markdown_report(dataset_name, summary, generated_plots, llm_insights, output_dir):
    """Compiles outputs into a final Markdown report."""
    md = f"# Data Profiling Report: {dataset_name}\n\n"
    md += f"## Dataset Overview\n- **Rows:** {summary['overview']['rows']:,}\n- **Columns:** {summary['overview']['columns']:,}\n\n"
    md += "## Data Quality Findings\n" + ("\n".join([f"- ⚠️ {w}" for w in summary['quality_warnings']]) if summary['quality_warnings'] else "- No major issues detected.") + "\n\n"
    md += f"## AI-Assisted Insights\n{llm_insights}\n\n"
    md += "## Visualizations\n" + "\n\n".join([f"![{plot}](plots/{plot})" for plot in generated_plots])
    
    with open(os.path.join(output_dir, 'report.md'), 'w', encoding='utf-8') as f: f.write(md)

In [ ]:
def generate_profile(csv_path, output_dir, use_llm=True):
    """Main Orchestrator Entry Point"""
    print(f"Starting analysis for {csv_path}...")
    dataset_name = os.path.basename(csv_path)
    os.makedirs(os.path.join(output_dir, 'plots'), exist_ok=True)
    
    df = pd.read_csv(csv_path)
    roles = infer_column_roles(df)
    
    # Execute the modular analysis functions
    overview = get_dataset_overview(df, roles)
    quality = get_data_quality_profile(df, overview, roles)
    stats = get_descriptive_statistics(df, roles)
    relationships = get_relationships_between_variables(df, roles)
    
    # Combine into one master JSON for the LLM
    summary = {
        'overview': overview,
        'quality_warnings': quality,
        'descriptive_statistics': stats,
        'relationships': relationships
    }
    
    with open(os.path.join(output_dir, 'analysis_summary.json'), 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=4)
        
    pd.DataFrame.from_dict(overview['column_info'], orient='index').to_csv(os.path.join(output_dir, 'column_profile.csv'), index_label='Column_Name')
    
    generated_plots = generate_adaptive_visualizations(df, roles, os.path.join(output_dir, 'plots'))
    llm_insights = run_llm_insights(summary, output_dir, use_llm)
    build_markdown_report(dataset_name, summary, generated_plots, llm_insights, output_dir)
    print(f"Success! Report generated at: {os.path.join(output_dir, 'report.md')}")